# Event-related CSD Univariate Analysis Pipeline

---

**Platforms and Packages Used:**
* **Python Version:** 3.11.8 (Anaconda 24.3.0)
* **NumPy Version:** 1.26.4
* **Pandas Version:** 2.3.2
* **SciPy Version:** 1.13.1
* **Pingouin Version:** 0.5.5
* **MNE-Python Version:** 1.10.0

**Terminology Mapping (Code vs. Manuscript):**

| Code / Filename | Manuscript Terminology |
| :--- | :--- |
| `"subject"` | "participant" |
| `"decisions"` | "action conditions" |
| `"PreAssn"` | "Pre-association phase" |
| `"Sess1"` | "Post-association Session1" |
| `"Sess2"` | "Post-association Session2" |

**Overview:**

This notebook performs univariate analyses on Current Source Density (CSD) data, integrating both the **Pre-Action** and **Post-Action** periods.
* **Pre-Action Phase:** Compares post-association sessions (both stimulus-present and stimulus-omitted trials) against the pre-association baseline within the pre-action time window.
* **Post-Action Phase:** Compares only stimulus-omitted trials in post-association sessions and compares them against the pre-association baseline within the post-action (e.g. post-stimulus onset) window,to isolate omission-evoked prediction errors.

The workflow has been divided into two analytical approaches:
* **Approach 1: Hypothesis-Driven (Predefined ROIs & Time windows)**
   * Extracts mean amplitudes directly from *priori* defined spatial-temporal ROIs (e.g., Occipital channels within specific time windows).
   * Applies rigorous statistical checks (Shapiro-Wilk for normality, Levene/Bartlett for homogeneity).
   * Executes Paired t-tests or Wilcoxon signed-rank tests with Hedges' g effect sizes.
   * False Discovery Rate (FDR) correction is applied independently for the *Cued* and *Voluntary* conditions.
* **Approach 2: Data-Driven Exploratory (Spatio-Temporal Permutation)**
   * Runs independently across all channels and time points.
   * Utilizes Mike X. Cohen's cluster-based permutation testing framework (Cohen, 2017 p.233-244) to intrinsically control for multiple comparisons across space and time.

**Pipeline Steps:**
1. **Environment Setup & Parameters:** Load required packages, define working directories, set up experimental parameters, and configure spatial adjacency.
2. **Centralized Data Loading:** Dual-stream loading into memory. Separately constructs `dict_preaction` and `dict_postaction` based on their distinct file templates and time-window slices.
3. **Approach 1 (Hypothesis-Driven):** Tests a-priori specified spatio-temporal ROIs for both Pre-Action and Post-Action periods. Applies independent False Discovery Rate (FDR) corrections for each Action Condition (Cued/Voluntary) and period.
4. **Approach 2 (Data-Driven Exploratory):** Executes spatio-temporal cluster-based permutation tests across both periods to organically discover neural patterns.

## Environmental Setup & Parameters

In [2]:
import os
import sys
import mne
import numpy as np
import pandas as pd
import pingouin as pg
from scipy import stats
from pathlib import Path
from itertools import product
from joblib import Parallel, delayed

# Set standard output formats
mne.set_log_level('CRITICAL')
np.set_printoptions(precision=3, linewidth=120)

# Define paths
# root_dir = Path('/path/to/your/project/directory') # Update with your actual path
# data_dir = root_dir / 'PipelineData/csddata'
# stat_dir = root_dir / 'statdata'
root_dir = Path('/home/zhengting/Insync/OneDrive/ParisCite/1_MultivariateProcedure/')
data_dir = Path('/home/zhengting/BackupData/ParisCite/1_MultivariateProcedure/PipelineData_3rd/amplstat_new_noout_interbdchs_CSD/')
stat_dir = root_dir / 'code/Github_pub/statdata'

# Experimental factors
# days = ['Day1', 'Day2']
decisions = ['Cued', 'Voluntary']
stimuli = ['Visual', 'Auditory']
sessions = ['PreAssn', 'Sess1', 'Sess2']
periods = ['PreAction', 'PostAction']
eegtype = 'CSD'

# Participant inclusion dict
num_subs = 20
dict_included_sids = {
    'Cued': np.arange(1, 21),
    # for univariate anlysis, S15 was excluded in Voluntary condition,
    # and details see in Table S1 and AEP_02 (cell of 'Sample Selection')
    'Voluntary': np.concatenate((np.arange(1, 15), np.array([16, 17, 18, 19, 20])))    
}

# Load metadata (Channels & Times) from a template file
# template_path = data_dir / '1/s1_Day1_CuedAuditory_CSD.set'
template_path = data_dir / '1/s1_Session1_CuedAuditory_CSD_AmplState.set'
data_raw_template = mne.io.read_epochs_eeglab(template_path, verbose='CRITICAL')
ch_names = np.asarray(data_raw_template.info['ch_names'])
times = data_raw_template.times
times_ms = (times * 1000).astype(int)

# Define period-specific time windows 
time_start_idx = 0
time_zero_idx  = 475
time_end_idx   = 725 # len(times)-1
times_ms_pre = times_ms[time_start_idx : time_zero_idx + 1]
times_ms_post = times_ms[time_zero_idx : time_end_idx + 1]

# Spatial Adjacency (For Data-driven clustering)
from mne.channels import read_ch_adjacency
adjacency, _ = read_ch_adjacency('easycapM1', picks=ch_names)
hubs, conns = np.where(adjacency.toarray())
electrode_neighbors = [conns[np.logical_and(hubs == i, conns != i)] for i in range(len(ch_names))]

## Centralized Data Loading

In [3]:
# Define file templates for Pre-Action vs Post-Action
# Pre-Action includes both Stim-present and Stim-omitted PostAssn trials, Post-Action only includes Stim-omitted PostAssn trials.

# templates_preaction = {
#     'PreAssn': ['data_folder/sid/ssid_Day1_decisionNostimulusStimPre_eegtype.set'],
#     'Sess1':   ['data_folder/sid/ssid_Day1_decisionNostimulusStimPost_eegtype.set',
#                 'data_folder/sid/ssid_Day1_decisionstimulus_eegtype.set'],
#     'Sess2':   ['data_folder/sid/ssid_Day2_decisionNostimulusStimPost_eegtype.set',
#                 'data_folder/sid/ssid_Day2_decisionstimulus_eegtype.set']
# }
# templates_postaction = {
#     'PreAssn': ['data_folder/sid/ssid_Day1_decisionNostimulusStimPre_eegtype.set'],
#     'Sess1':   ['data_folder/sid/ssid_Day1_decisionNostimulusStimPost_eegtype.set'],
#     'Sess2':   ['data_folder/sid/ssid_Day2_decisionNostimulusStimPost_eegtype.set']
# }
templates_preaction = {
    'PreAssn': ['data_folder/sid/ssid_Session1_decisionNostimulusStimPre_eegtype_AmplState.set'],
    'Sess1':   ['data_folder/sid/ssid_Session1_decisionNostimulusStimPost_eegtype_AmplState.set',
                'data_folder/sid/ssid_Session1_decisionstimulus_eegtype_AmplState.set'],
    'Sess2':   ['data_folder/sid/ssid_Session2_decisionNostimulusStimPost_eegtype_AmplState.set',
                'data_folder/sid/ssid_Session2_decisionstimulus_eegtype_AmplState.set']
}
templates_postaction = {
    'PreAssn': ['data_folder/sid/ssid_Session1_decisionNostimulusStimPre_eegtype_AmplState.set'],
    'Sess1':   ['data_folder/sid/ssid_Session1_decisionNostimulusStimPost_eegtype_AmplState.set'],
    'Sess2':   ['data_folder/sid/ssid_Session2_decisionNostimulusStimPost_eegtype_AmplState.set']
}

def resolve_paths(template_list, sid, decis, stim):
    return [tpl.replace('data_folder', str(data_dir))
               .replace('sid', str(sid))
               .replace('decision', decis)
               .replace('stimulus', stim)
               .replace('eegtype', eegtype) for tpl in template_list]

# Initialize dual storage dictionaries
dict_preaction = {d: {s: {} for s in stimuli} for d in decisions}
dict_postaction = {d: {s: {} for s in stimuli} for d in decisions}

print("Centralizing EEG data into memory for BOTH phases...")

for decis, stim in product(decisions, stimuli):
    included_sids = dict_included_sids[decis]
    
    for sess in sessions:
        # 1. Load Pre-Action Data
        data_pre = []
        for sid in included_sids:
            paths = resolve_paths(templates_preaction[sess], sid, decis, stim)
            trials = [mne.io.read_epochs_eeglab(p, verbose='CRITICAL').get_data(copy=True) for p in paths if Path(p).exists()]
            if trials: data_pre.append(np.concatenate(trials).mean(axis=0, keepdims=True))
        
        if data_pre:
            stacked_pre = np.transpose(np.concatenate(data_pre), (0, 2, 1))
            dict_preaction[decis][stim][sess] = stacked_pre[:, time_start_idx : time_zero_idx + 1, :]
            
        # 2. Load Post-Action Data
        data_post = []
        for sid in included_sids:
            paths = resolve_paths(templates_postaction[sess], sid, decis, stim)
            trials = [mne.io.read_epochs_eeglab(p, verbose='CRITICAL').get_data(copy=True) for p in paths if Path(p).exists()]
            if trials: data_post.append(np.concatenate(trials).mean(axis=0, keepdims=True))
            
        if data_post:
            stacked_post = np.transpose(np.concatenate(data_post), (0, 2, 1))
            dict_postaction[decis][stim][sess] = stacked_post[:, time_zero_idx : time_end_idx + 1, :]

print("Dual-stream data loading complete.")

Centralizing EEG data into memory for BOTH phases...
Dual-stream data loading complete.


## Hypothesis-Driven Analysis

Iterates through Pre-Action and Post-Action periods sequentially, and applies independent FDR adjustments for each [Period] x [Decision].

In [7]:
def compute_roi_amplitude(data_cube, times_vector, ch_list, time_bounds, target_chans):
    t_mask = (times_vector >= time_bounds[0]) & (times_vector <= time_bounds[1])
    ch_mask = np.isin(ch_list, target_chans)
    return data_cube[:, t_mask, :][:, :, ch_mask].mean(axis=(1, 2))

roi_configs = {
    'PreAction': {
        'Visual': {'chans': ['O1', 'Oz', 'O2'], 'times': [-50, 0]}, # Occipital, Late stage for both Cued and Voluntary
        'Auditory': {'chans': ['T8','CP6','C6','TP8'], 'times': [-150, 0]}, # Temporal, Late stage for both Cued and Voluntary
        'Voluntary_Visual': {'chans': ['O1', 'Oz', 'O2'], 'times': [-400, -200]}, # Early stage only for Voluntary
        'Voluntary_Auditory': {'chans': ['T8','CP6','C6','TP8'], 'times': [-400, -200]}, # Early stage only for Voluntary
    },
    'PostAction': {
        'Visual': {'chans': ['O1', 'Oz', 'O2'], 'times': [80, 200]},
        'Cued_Auditory': {'chans': ['T7','CP5','C5','TP7','T8','CP6','C6','TP8'], 'times': [80, 200]},
        'Voluntary_Auditory_R': {'chans': ['T8','CP6','C6','TP8'], 'times': [80, 200]},
        'Voluntary_Auditory_L': {'chans': ['T7','CP5','C5','TP7'], 'times': [80, 200]}
    }
}
def tag_significance(row):
    p_unc = row['p_unc']
    p_fdr = row['p_fdr']
    if p_fdr < 0.05: return 'Sign.'
    if p_unc < 0.05 and p_fdr < 0.10: return 'Margin. Sign.'
    return 'Non-Sign.'

for period in periods:
    current_dict = dict_preaction if period == 'PreAction' else dict_postaction
    current_times = times_ms_pre if period == 'PreAction' else times_ms_post
    
    for decis in decisions:
        print(f"\n{'='*85}\n [HYPOTHESIS-DRIVEN] PERIOD: {period.upper()} | DECISION: {decis.upper()}\n{'='*85}")
        
        roi_stat_accumulator = []
        
        for stim in stimuli:
            applicable_rois = {}
            if period == 'PreAction':
                applicable_rois[f'{decis}_{stim}_Late'] = roi_configs[period][f'{stim}']
                if decis == 'Voluntary': applicable_rois[f'{decis}_{stim}_Early'] = roi_configs[period][f'{decis}_{stim}']
            else: # PostAction
                if stim == 'Visual': applicable_rois[f'{decis}_{stim}'] = roi_configs[period]['Visual']
                elif stim == 'Auditory':
                    if decis == 'Cued':
                        applicable_rois[f'{decis}_{stim}'] = roi_configs[period][f'{decis}_{stim}']
                    else:
                        applicable_rois[f'{decis}_{stim}_L'] = roi_configs[period][f'{decis}_{stim}_L']
                        applicable_rois[f'{decis}_{stim}_R'] = roi_configs[period][f'{decis}_{stim}_R']
                
            for roi_name, params in applicable_rois.items():
                for post_sess in ['Sess1', 'Sess2']:
                    
                    cube_pre = current_dict[decis][stim]['PreAssn']
                    cube_post = current_dict[decis][stim][post_sess]
                    
                    mean_pre = compute_roi_amplitude(cube_pre, current_times, ch_names, params['times'], params['chans'])
                    mean_post = compute_roi_amplitude(cube_post, current_times, ch_names, params['times'], params['chans'])
                    diff_vector = mean_post - mean_pre
                    
                    # Statistical Testing
                    normality = pg.normality(diff_vector, method='shapiro')
                    is_normal = normality['normal'].values[0]
                    _, p_levene = stats.levene(mean_post, mean_pre)
                    _, p_bartlett = stats.bartlett(mean_post, mean_pre)
                    is_homogenous = (p_levene > 0.05) or (p_bartlett > 0.05)
                    
                    if is_normal and is_homogenous:
                        test_res = pg.ttest(mean_post, mean_pre, paired=True, alternative='two-sided')
                        t_val, p_unc, method = test_res['T'].values[0], test_res['p-val'].values[0], 't-test'
                    else:
                        test_res = pg.wilcoxon(mean_post, mean_pre, alternative='two-sided')
                        t_val, p_unc, method = test_res['W-val'].values[0], test_res['p-val'].values[0], 'Wilcoxon'
                        
                    h_g = pg.compute_effsize(mean_post, mean_pre, paired=True, eftype='hedges')
                    
                    roi_stat_accumulator.append({
                        'Stimulus': stim, 'ROI_Name': roi_name, 'Comparison': f"{post_sess} vs Pre",
                        'Method': method, 'Stat': t_val, 'Shap_p': normality['pval'].values[0],
                        'Lev_p': p_levene, 'Bart_p': p_bartlett, 'p_unc': p_unc, 'Hedges_g': h_g
                    })
                    
        # Apply independent FDR calibration for this specific period + Decision combination
        if roi_stat_accumulator:
            df_roi_results = pd.DataFrame(roi_stat_accumulator)
            h_reject, p_fdr = pg.multicomp(df_roi_results['p_unc'].values, method='fdr_bh')
            df_roi_results['p_fdr'] = p_fdr
            df_roi_results['Sign.'] = df_roi_results.apply(tag_significance, axis=1)
            
            print(df_roi_results.to_string(index=False, formatters={
                'Stat': '{:0.3f}'.format, 'Shap_p': '{:0.4f}'.format, 'Lev_p': '{:0.4f}'.format, 'Bart_p': '{:0.4f}'.format,
                'p_unc': '{:0.4f}'.format, 'p_fdr': '{:0.4f}'.format, 'Hedges_g': '{:0.3f}'.format
            }))


 [HYPOTHESIS-DRIVEN] PERIOD: PREACTION | DECISION: CUED
Stimulus           ROI_Name   Comparison Method   Stat Shap_p  Lev_p Bart_p  p_unc Hedges_g  p_fdr         Sign.
  Visual   Cued_Visual_Late Sess1 vs Pre t-test  2.380 0.5774 0.0847 0.1294 0.0279    0.637 0.0558 Margin. Sign.
  Visual   Cued_Visual_Late Sess2 vs Pre t-test  0.894 0.5171 0.2803 0.3383 0.3823    0.276 0.3823     Non-Sign.
Auditory Cued_Auditory_Late Sess1 vs Pre t-test -4.111 0.0767 0.8036 0.9173 0.0006   -0.671 0.0024         Sign.
Auditory Cued_Auditory_Late Sess2 vs Pre t-test -2.021 0.3521 0.7290 0.7549 0.0575   -0.436 0.0767     Non-Sign.

 [HYPOTHESIS-DRIVEN] PERIOD: PREACTION | DECISION: VOLUNTARY
Stimulus                 ROI_Name   Comparison   Method   Stat Shap_p  Lev_p Bart_p  p_unc Hedges_g  p_fdr     Sign.
  Visual    Voluntary_Visual_Late Sess1 vs Pre   t-test  0.596 0.9905 0.4564 0.4977 0.5589    0.144 0.6799 Non-Sign.
  Visual    Voluntary_Visual_Late Sess2 vs Pre Wilcoxon 69.000 0.5820 0.0459 0.027

## Data-Driven Exploratory Analysis
With Trimming

In [5]:
from AEP_03a_permutation_stats import run_cohen_spatiotemporal_permutation

n_permutations = 10000
alpha_cluster_forming = 0.02
cluster_p_threshold = 0.05
n_seed = 0 # Reproducibility seed

# Trimming parameters
ratio_ch_sel = 0.15
min_distinct_chans = 2
time_res = 10 # ms

# Confound exclusion limits
preaction_confound_limit = -500 # ms (Exclude clusters starting earlier than this to avoid tactile prime)
postaction_confound_limit = 300 # ms (Exclude clusters ending later than this to avoid late cognitive confounds)

dict_permutation_outputs = {}

print(f"Initiating Exploratory Permutation Tests (n_perms={n_permutations}, seed={n_seed})...\n")
for period in periods[:]:
    print(f"\n{'='*50}\n PROCESSING PERIOD: {period.upper()} (WITH TRIMMING) \n{'='*50}")
    current_dict = dict_preaction if period == 'PreAction' else dict_postaction
    current_times = times_ms_pre if period == 'PreAction' else times_ms_post
    
    for decis, stim in product(decisions[:], stimuli[:]):
        for post_sess in ['Sess1', 'Sess2']:
            comparison_id = f"{period}_{decis}_{stim}_{post_sess}_vs_Pre"
            print(f"\n--> Analyzing: {comparison_id}")
            
            cube_pre = current_dict[decis][stim]['PreAssn']
            cube_post = current_dict[decis][stim][post_sess]
            true_diffs = cube_post - cube_pre
            
            # Execute the core permutation engine
            t_map, all_clusters, clu_stats, null_dists = run_cohen_spatiotemporal_permutation(
                true_diffs, n_permutations=n_permutations, 
                alpha_forming=alpha_cluster_forming, 
                topology_map=electrode_neighbors, random_seed=n_seed
            )

            sig_mask = clu_stats['pval_sumstats'] < cluster_p_threshold

            if not sig_mask.any():
                print("    (No initial significant clusters found)")
                continue
            
            # Process only the clusters that passed the initial significance pre-check
            filtered_coords = [all_clusters[idx] for idx in clu_stats.index[sig_mask]]
            filtered_pvals = clu_stats.loc[sig_mask, 'pval_sumstats'].values
            
            surviving_count = 0
            surviving_coords = []
            for clu_coords, p_raw in zip(filtered_coords, filtered_pvals):
                clu_array = np.array(clu_coords)
                if len(clu_array) == 0: continue

                # --- TRIMMING STEP 0: Temporal Confound Checks ---
                t_idx_initial = np.unique(clu_array[:, 1])
                t_start_initial_ms = current_times[t_idx_initial[0]]
                t_end_initial_ms = current_times[t_idx_initial[-1]]
                
                if period == 'PreAction' and t_start_initial_ms < preaction_confound_limit:
                    # Skip cluster entirely as it enters the tactile prime confound zone (< -500ms)
                    continue
                if period == 'PostAction' and t_end_initial_ms > postaction_confound_limit:
                    # Skip cluster entirely as it extends into late cognitive processing zone (> 300ms)
                    continue
                
                # --- TRIMMING STEP 1: Ensure at least 2 distinct channels remain ---
                if len(np.unique(clu_array[:, 0])) < min_distinct_chans:
                    continue
                
                # --- TRIMMING STEP 2: Remove electrodes present for < 20% of duration ---
                original_duration = len(np.unique(clu_array[:, 1]))
                ch_idx, ch_counts = np.unique(clu_array[:, 0], return_counts=True)
                valid_chans = ch_idx[ch_counts / original_duration > ratio_ch_sel]
                clu_array = clu_array[np.isin(clu_array[:, 0], valid_chans)]
                if len(clu_array) == 0: continue
                
                # --- TRIMMING STEP 3: Adjust temporal boundaries to 10ms resolution ---
                t_idx = np.unique(clu_array[:, 1])
                t_start_ms, t_end_ms = current_times[t_idx[0]], current_times[t_idx[-1]]
                
                # Conservative 10ms bounding 
                t_start_adj = np.ceil(t_start_ms / time_res) * time_res
                t_end_adj = np.floor(t_end_ms / time_res) * time_res
                
                valid_t_mask = (current_times >= t_start_adj) & (current_times <= t_end_adj)
                valid_t_idx = np.where(valid_t_mask)[0]
                clu_array = clu_array[np.isin(clu_array[:, 1], valid_t_idx)]
                if len(clu_array) == 0: continue
                        
                # --- TRIMMING STEP 4: Re-evaluate 20% rule after time crop ---
                new_duration = len(np.unique(clu_array[:, 1]))
                if new_duration == 0: continue
                ch_idx, ch_counts = np.unique(clu_array[:, 0], return_counts=True)
                valid_chans = ch_idx[ch_counts / new_duration > ratio_ch_sel]
                clu_array = clu_array[np.isin(clu_array[:, 0], valid_chans)]
                if len(clu_array) == 0: continue
                
                # --- Safety Check: Ensure >= 2 distinct channels STILL remain after all trimming ---
                final_chans = np.unique(clu_array[:, 0])
                if len(final_chans) < min_distinct_chans: 
                    continue
                
                # --- RE-EVALUATION: Calculate updated P-value & Hedges' g ---
                new_mass = np.sum([t_map[t, c] for c, t in clu_array])
                c_dist_idx = 2 if new_mass > 0 else 3 # 2nd column for sumstats of positive clusters; 3rd for negativity
                p_updated = (np.sum(np.abs(null_dists[:, c_dist_idx]) > np.abs(new_mass)) + 1) / (n_permutations + 1)
                
                if p_updated < cluster_p_threshold:
                    surviving_count += 1
                    surviving_coords.append(clu_array)
                    
                    # Compute Hedges' g 
                    mask = np.zeros((len(current_times), len(ch_names)), dtype=bool)
                    for c, t in clu_array: mask[t, c] = True
                
                    subj_mean_pre = np.mean(cube_pre[:, mask], axis=1)
                    subj_mean_post = np.mean(cube_post[:, mask], axis=1)
                    h_g = pg.compute_effsize(subj_mean_post, subj_mean_pre, paired=True, eftype='hedges')
                    
                    # Print layout
                    ch_freq = ch_counts[np.isin(ch_idx, valid_chans)] / new_duration
                    chan_freq_str = ", ".join([f"{ch_names[c]}({f:.2f})" for c, f in zip(final_chans, ch_freq)])
                    
                    print(f"    [Surviving Cluster #{surviving_count}] | p_raw = {p_raw:.4f} -> p_updated = {p_updated:.4f} | g = {h_g:.3f}")
                    print(f"      - Times: {int(t_start_adj)} ~ {int(t_end_adj)} ms")
                    print(f"      - Channels: {chan_freq_str}")

            if surviving_count == 0:
                print("    (Clusters found, but none survived the strict trimming constraints)")
            else:
                dict_permutation_outputs[comparison_id] = {
                    't_map': t_map,
                    'sig_clusters': surviving_coords
                }

print("\nExploratory Permutation Testing Complete.")
save_path = stat_dir / 'dict_permutation_outputs.npy'
np.save(save_path, dict_permutation_outputs, allow_pickle=True)
print(f"\n[INFO] Data-driven permutation results successfully saved to:\n -> {save_path}")

Initiating Exploratory Permutation Tests (n_perms=10000, seed=0)...


 PROCESSING PERIOD: PREACTION (WITH TRIMMING) 

--> Analyzing: PreAction_Cued_Visual_Sess1_vs_Pre
    [Surviving Cluster #1] | p_raw = 0.0117 -> p_updated = 0.0140 | g = -0.877
      - Times: -280 ~ -120 ms
      - Channels: F3(1.00), FC1(0.22), F1(0.57), FC3(0.43)

--> Analyzing: PreAction_Cued_Visual_Sess2_vs_Pre
    [Surviving Cluster #1] | p_raw = 0.0227 -> p_updated = 0.0304 | g = -1.156
      - Times: -370 ~ -230 ms
      - Channels: F7(0.20), F3(0.82), AF3(0.28), F5(0.39), F1(0.62)

--> Analyzing: PreAction_Cued_Auditory_Sess1_vs_Pre
    [Surviving Cluster #1] | p_raw = 0.0075 -> p_updated = 0.0089 | g = -0.992
      - Times: -380 ~ -250 ms
      - Channels: Fz(1.00), FC1(0.38), F1(0.94)
    [Surviving Cluster #2] | p_raw = 0.0038 -> p_updated = 0.0046 | g = -0.741
      - Times: -150 ~ 0 ms
      - Channels: T8(0.22), CP6(0.62), C6(1.00), TP8(0.67)
    [Surviving Cluster #3] | p_raw = 0.0025 -> p_updated = 0.